# Customer Service Agent on Databricks
## Claims-support assistant using your existing `claims` Unity Catalog

This notebook builds a tool-using customer service agent over your five inspected tables:

- `claims.analytics.fact_claims` — 3,046 claims
- `claims.analytics.dim_member` — 500 members
- `claims.analytics.dim_policy` — 661 policies
- `claims.analytics.dim_plan` — 4 plans
- `claims.analytics.dim_hospital` — 40 hospitals

Your earlier integrity check reported 3,046 claim rows and zero missing member, policy, plan, or hospital joins.

**Learning goals:** Unity Catalog and Spark access, Databricks Model Serving, Python tools, LangChain agent orchestration, MLflow tracing, and basic evaluation.

> **Security:** `MEM-0317` is a fixed demo identity, not authentication. A real app must authenticate the customer in a trusted backend and pass the verified identity to tools. Never trust a member ID typed into chat.

## 1. Architecture — what each part does

| Layer | Technology | Responsibility |
|---|---|---|
| Data | Unity Catalog + Spark | Governed tables and queries |
| Model | Databricks Model Serving | Hosts the LLM endpoint |
| Tools | Python + Spark | Controlled business lookups |
| Agent | LangChain `create_agent` | Model selects and calls tools |
| Observability | MLflow | Experiments, metrics, traces |

Databricks provides the data platform, Unity Catalog, Model Serving, and MLflow. LangChain is the orchestration framework in this notebook.

**Request flow:** Customer question → LLM decides whether a tool is needed → LangChain executes the tool → Spark queries Unity Catalog → tool result returns to LLM → natural-language answer.

## 2. Install dependencies\nRun once. If Databricks requests a Python restart, run the next cell and then continue from imports.

In [0]:
%pip install -U databricks-langchain langchain langgraph mlflow databricks-ai-search "langgraph-prebuilt==1.0.8"

Looking in indexes: [REDACTED]
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langchain/1.5.0/langchain-1.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached http://package-proxy.host.local:8184/pypi/v1/packages/langgraph/1.2.22/langgraph-1.2.22-py3-none-any.whl.metadata (4.9 kB)
INFO: pip is looking at multiple versions of langgraph to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.13
    Uninstalling langgraph-prebuilt-1.0.13:
      Suc

In [0]:
dbutils.library.restartPython()

## 3. Imports\nSpark is provided by the Databricks runtime. `ChatDatabricks` connects LangChain to Model Serving; `create_agent` builds the tool-calling loop; MLflow tracks the application.

In [0]:
import json
import pandas as pd
from pyspark.sql import functions as F

from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langchain.agents import create_agent
import mlflow

print("Imports complete")
print("Databricks user:", spark.sql("SELECT current_user()").first()[0])

Imports complete
Databricks user: kiritjain@cyntexa.com


## 4. MLflow experiment\nAn experiment groups runs, parameters, metrics, artifacts, and traces. We keep this agent separate from the External Tools Research Agent.

In [0]:
CURRENT_USER = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT_NAME = f"/Users/{CURRENT_USER}/Customer_Service_Agent"
mlflow.set_experiment(EXPERIMENT_NAME)
print(EXPERIMENT_NAME)

/Users/kiritjain@cyntexa.com/Customer_Service_Agent


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## 5. Configure and check Unity Catalog tables\nA three-part name is `catalog.schema.table`. Unity Catalog controls access to these objects. This cell prints counts, not personal data.

In [0]:
TABLES = {
    "claims": "claims.analytics.fact_claims",
    "members": "claims.analytics.dim_member",
    "policies": "claims.analytics.dim_policy",
    "plans": "claims.analytics.dim_plan",
    "hospitals": "claims.analytics.dim_hospital"
}
for label, table_name in TABLES.items():
    print(f"{label:12} | {table_name:40} | {spark.table(table_name).count():,} rows")

claims       | claims.analytics.fact_claims             | 3,046 rows
members      | claims.analytics.dim_member              | 500 rows
policies     | claims.analytics.dim_policy              | 661 rows
plans        | claims.analytics.dim_plan                | 4 rows
hospitals    | claims.analytics.dim_hospital            | 40 rows


## 6. Verify dimension joins\nThe fact table uses surrogate keys to connect each claim to its member, policy, plan, and hospital. This validates those relationships before the agent uses them.

In [0]:
f = spark.table(TABLES["claims"]).alias("f")
m = spark.table(TABLES["members"]).alias("m")
p = spark.table(TABLES["policies"]).alias("p")
pl = spark.table(TABLES["plans"]).alias("pl")
h = spark.table(TABLES["hospitals"]).alias("h")

joined = (f
    .join(m, F.col("f.member_sk") == F.col("m.member_sk"), "left")
    .join(p, F.col("f.policy_sk") == F.col("p.policy_sk"), "left")
    .join(pl, F.col("f.plan_sk") == F.col("pl.plan_sk"), "left")
    .join(h, F.col("f.hospital_sk") == F.col("h.hospital_sk"), "left"))

display(joined.agg(
    F.count("*").alias("total_claim_rows"),
    F.sum(F.when(F.col("m.member_sk").isNull(), 1).otherwise(0)).alias("missing_member"),
    F.sum(F.when(F.col("p.policy_sk").isNull(), 1).otherwise(0)).alias("missing_policy"),
    F.sum(F.when(F.col("pl.plan_sk").isNull(), 1).otherwise(0)).alias("missing_plan"),
    F.sum(F.when(F.col("h.hospital_sk").isNull(), 1).otherwise(0)).alias("missing_hospital")
))

total_claim_rows,missing_member,missing_policy,missing_plan,missing_hospital
3046,0,0,0,0


## 7. Demo identity and model endpoint\nWe simulate an already-authenticated application session for `MEM-0317`, present in your sample records. Tools below do not let the LLM choose a member ID. **This fixed variable is only for demonstration; it is not production authentication.**

In [0]:
DEMO_VERIFIED_MEMBER_ID = "MEM-0317"
MODEL_ENDPOINT = "databricks-gpt-oss-20b"  # Change to an enabled endpoint in your workspace
print("Demo member:", DEMO_VERIFIED_MEMBER_ID)
print("Model endpoint:", MODEL_ENDPOINT)

Demo member: MEM-0317
Model endpoint: databricks-gpt-oss-20b


## 8. Connect to Databricks Model Serving\n`ChatDatabricks` is a LangChain wrapper. The model runs behind the serving endpoint; this notebook sends a request and receives the response. Test it before building the agent.

In [0]:
llm = ChatDatabricks(endpoint=MODEL_ENDPOINT, temperature=0.1, max_tokens=700)
model_test = llm.invoke("Explain in one sentence what a health insurance claim status means.")
print(model_test.content)

A health‑insurance claim status indicates the current processing stage of a submitted claim—whether it’s pending, approved, denied, or needs additional information.


## 9. Tool 1 — claim details\nA tool is a Python function with a defined purpose and input/output contract. `@tool` exposes its name, description, and argument schema to the LLM. This query filters by both claim ID and the fixed demo member ID, joins dimensions, and returns only support-relevant fields.

In [0]:
@tool
def get_claim_details(claim_id: str) -> str:
    """Get claim status, dates, amounts, plan, and hospital for the verified demo member."""
    claim = spark.table(TABLES["claims"]).alias("f")
    member = spark.table(TABLES["members"]).alias("m")
    policy = spark.table(TABLES["policies"]).alias("p")
    plan = spark.table(TABLES["plans"]).alias("pl")
    hospital = spark.table(TABLES["hospitals"]).alias("h")

    rows = (claim
        .filter((F.col("f.claim_id") == claim_id) &
                (F.col("f.member_id") == DEMO_VERIFIED_MEMBER_ID))
        .join(member, F.col("f.member_sk") == F.col("m.member_sk"), "left")
        .join(policy, F.col("f.policy_sk") == F.col("p.policy_sk"), "left")
        .join(plan, F.col("f.plan_sk") == F.col("pl.plan_sk"), "left")
        .join(hospital, F.col("f.hospital_sk") == F.col("h.hospital_sk"), "left")
        .select(
            F.col("f.claim_id").alias("claim_id"),
            F.col("f.status").alias("status"),
            F.col("f.rejection_code").alias("rejection_code"),
            F.col("f.billed_amount").alias("billed_amount"),
            F.col("f.approved_amount").alias("approved_amount"),
            F.col("f.admission_date").alias("admission_date"),
            F.col("f.discharge_date").alias("discharge_date"),
            F.col("f.submitted_ts").alias("submitted_at"),
            F.col("f.settled_ts").alias("settled_at"),
            F.col("p.policy_id").alias("policy_id"),
            F.col("pl.plan_code").alias("plan_code"),
            F.col("pl.sum_insured").alias("sum_insured"),
            F.col("pl.room_rent_cap_per_day").alias("room_rent_cap_per_day"),
            F.col("h.name").alias("hospital_name"),
            F.col("h.city").alias("hospital_city"))
        .limit(1).collect())

    if not rows:
        return "No matching claim was found for the currently verified member. Do not reveal whether it exists for another member."
    return json.dumps(rows[0].asDict(), default=str)

## 10. Tool 2 — current policy and plan benefits\nThis finds current policies for the verified member and joins the plan dimension. These structured fields do not necessarily represent every policy clause, exclusion, or exception.

In [0]:
@tool
def get_member_policies() -> str:
    """Get current policies and plan benefit fields for the verified demo member."""
    policy = spark.table(TABLES["policies"]).alias("p")
    plan = spark.table(TABLES["plans"]).alias("pl")
    rows = (policy
        .filter((F.col("p.member_id") == DEMO_VERIFIED_MEMBER_ID) &
                (F.col("p.is_current") == True))
        .join(plan, F.col("p.plan_code") == F.col("pl.plan_code"), "left")
        .select(
            F.col("p.policy_id").alias("policy_id"),
            F.col("p.plan_code").alias("plan_code"),
            F.col("p.valid_from").alias("valid_from"),
            F.col("p.valid_to").alias("valid_to"),
            F.col("pl.plan_family").alias("plan_family"),
            F.col("pl.sum_insured").alias("sum_insured"),
            F.col("pl.room_rent_cap_per_day").alias("room_rent_cap_per_day"),
            F.col("pl.copay_pct").alias("copay_pct"),
            F.col("pl.ped_waiting_months").alias("ped_waiting_months"),
            F.col("pl.initial_waiting_days").alias("initial_waiting_days"),
            F.col("pl.annual_premium").alias("annual_premium"))
        .limit(10).collect())
    if not rows:
        return "No current policy was found for the currently verified member."
    return json.dumps([r.asDict() for r in rows], default=str)

## 11. Tool 3 — hospital directory\nThis retrieves directory details by hospital ID. It does not expose member or claim records.

In [0]:
@tool
def get_hospital_info(hospital_id: str) -> str:
    """Look up a hospital's directory details using its hospital ID."""
    rows = (spark.table(TABLES["hospitals"])
        .filter(F.col("hospital_id") == hospital_id)
        .select("hospital_id", "name", "city", "city_tier", "state")
        .limit(1).collect())
    if not rows:
        return "No hospital was found with that hospital ID."
    return json.dumps(rows[0].asDict(), default=str)

## 12. Test tools independently\nTest the Python functions before involving the LLM. This separates query/data issues from model/agent issues. The claim ID below was in your sample data.

In [0]:
print("CLAIM TOOL")
print(get_claim_details.invoke({"claim_id": "CLM-10341"}))
print("\\nPOLICY TOOL")
print(get_member_policies.invoke({}))
print("\\nHOSPITAL TOOL")
print(get_hospital_info.invoke({"hospital_id": "HOS-001"}))

CLAIM TOOL
{"claim_id": "CLM-10341", "status": "submitted", "rejection_code": null, "billed_amount": 132605.57, "approved_amount": null, "admission_date": "2026-03-11", "discharge_date": "2026-03-19", "submitted_at": "2026-03-11 13:43:00", "settled_at": null, "policy_id": "POL-0317", "plan_code": "CORP_GROUP", "sum_insured": 500000, "room_rent_cap_per_day": 5000, "hospital_name": "Mumbai Care Hospital 11", "hospital_city": "Mumbai"}
\nPOLICY TOOL
[{"policy_id": "POL-0317", "plan_code": "CORP_GROUP", "valid_from": "2024-12-29", "valid_to": null, "plan_family": "group", "sum_insured": 500000, "room_rent_cap_per_day": 5000, "copay_pct": 0, "ped_waiting_months": 12, "initial_waiting_days": 0, "annual_premium": 8000}]
\nHOSPITAL TOOL
{"hospital_id": "HOS-001", "name": "Mumbai Care Hospital 01", "city": "Mumbai", "city_tier": "metro", "state": "Maharashtra"}


## 13. Register tools\nThe agent receives only these purpose-built functions, not unrestricted Spark access or arbitrary SQL execution.

In [0]:
customer_service_tools = [
    get_claim_details,
    get_member_policies,
    get_hospital_info
]
for t in customer_service_tools:
    print(t.name, "—", t.description)

get_claim_details — Get claim status, dates, amounts, plan, and hospital for the verified demo member.
get_member_policies — Get current policies and plan benefit fields for the verified demo member.
get_hospital_info — Look up a hospital's directory details using its hospital ID.


## 14. Optional — existing policy AI Search index\nYour earlier assignment created `claims.genai.policy_chunks` and index `claims.genai.policy_chunks_idx` on endpoint `filing_analyst_endpoint`. This optional section retrieves policy wording; the structured-data agent works without it.

In [0]:
POLICY_SEARCH_ENABLED = False
policy_index = None

try:
    # Databricks AI Search SDK (not the older VectorSearchClient import)
    from databricks.ai_search.client import AISearchClient

    ai_search_client = AISearchClient()
    policy_index = ai_search_client.get_index(
        index_name="claims.genai.policy_chunks_idx"
    )

    POLICY_SEARCH_ENABLED = True
    print("Connected to the existing Databricks AI Search index.")
    print("Index: claims.genai.policy_chunks_idx")

except Exception as e:
    print("Policy AI Search unavailable; continuing without it.")
    print(type(e).__name__ + ":", str(e)[:1000])


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Connected to the existing Databricks AI Search index.
Index: claims.genai.policy_chunks_idx


### Optional policy search tool

This tool queries the existing `claims.genai.policy_chunks_idx` index using **hybrid search**. Hybrid search combines semantic similarity with keyword matching.

The SDK returns a result object containing a manifest (column names) and result rows. The next cell converts those rows into readable text for the agent.


In [0]:
if POLICY_SEARCH_ENABLED:
    @tool
    def search_policy_documents(question: str) -> str:
        """Search the existing policy AI Search index and return relevant policy chunks."""
        response = policy_index.similarity_search(
            query_text=question,
            columns=["chunk_id", "doc_name", "heading", "text"],
            num_results=5,
            query_type="HYBRID"
        )

        # Convert the SDK response rows into dictionaries using the returned column names.
        column_names = [
            column["name"]
            for column in response["manifest"]["columns"]
        ]

        rows = response["result"]["data_array"]
        documents = [
            dict(zip(column_names, row))
            for row in rows
        ]

        return json.dumps(documents, ensure_ascii=False, default=str)

    customer_service_tools.append(search_policy_documents)
    print("Registered policy AI Search tool.")
else:
    print("Skipped optional policy tool.")


Registered policy AI Search tool.


## 15. Create the Customer Service Agent\n`create_agent` implements the tool loop: the model sees tool descriptions, chooses whether to call one, LangChain executes it, the result returns to the model, and the model writes the answer.

In [0]:
SYSTEM_PROMPT = f"""
You are a helpful customer service assistant for a health insurance claims service.
The application has verified the customer for this notebook session.
The only member whose private claim and policy data you may retrieve is {DEMO_VERIFIED_MEMBER_ID}.
The tools are bound to this verified context. Never try to select another member.

Rules:
- Use get_claim_details for claim-specific questions.
- Use get_member_policies for current policy and plan benefit questions.
- Use get_hospital_info for hospital directory questions.
- If search_policy_documents is available, use it for policy wording questions.
- Base factual answers on tool results. Say clearly when information is unavailable.
- Explain statuses without changing their meaning.
- Explain rejection codes only when the data or retrieved policy text supports it.
- Do not claim a claim is payable or covered unless retrieved information explicitly supports it.
- Do not provide medical advice or reveal another member's information.
- Never invent amounts, terms, dates, or hospital details.
- Keep responses concise and understandable.
"""

agent = create_agent(
    model=llm,
    tools=customer_service_tools,
    system_prompt=SYSTEM_PROMPT
)
print("Agent created. Tools:", [t.name for t in customer_service_tools])

Agent created. Tools: ['get_claim_details', 'get_member_policies', 'get_hospital_info', 'search_policy_documents']


## 16. First customer conversation\nAsk about the sample claim and observe whether the agent calls the claim tool.

In [0]:
question = "What is the current status of my claim CLM-10341, and how much was billed?"
response = agent.invoke({"messages": [{"role": "user", "content": question}]})
for message in response["messages"]:
    print("\\n", type(message).__name__)
    print(message)

\n HumanMessage
content='What is the current status of my claim CLM-10341, and how much was billed?' additional_kwargs={} response_metadata={} id='67422ffc-e6e4-4bdc-8e5e-9887c493e389'
\n AIMessage
content='' additional_kwargs={'tool_calls': [{'id': 'call_32a14aac-3f58-4b2b-a39a-e387032c067f', 'function': {'arguments': '{"claim_id":"CLM-10341"}', 'name': 'get_claim_details'}, 'type': 'function'}]} response_metadata={'usage': {'prompt_tokens': 446, 'completion_tokens': 46, 'total_tokens': 492}, 'prompt_tokens': 446, 'completion_tokens': 46, 'total_tokens': 492, 'model': 'gpt-oss-20b-080525', 'model_name': 'gpt-oss-20b-080525', 'finish_reason': 'tool_calls'} id='lc_run--01a1ed4c-31cb-74b5-b029-8b905b228489-0' tool_calls=[{'name': 'get_claim_details', 'args': {'claim_id': 'CLM-10341'}, 'id': 'call_32a14aac-3f58-4b2b-a39a-e387032c067f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 446, 'output_tokens': 46, 'total_tokens': 492, 'input_token_details': {'audio':

## 17. Inspect tool calls and final answer\nThis helper prints the tool names/arguments, tool results, and final assistant response from the returned message sequence.

In [0]:
def inspect_agent_response(agent_response):
    messages = agent_response.get("messages", [])
    print("TOOL CALLS")
    found = False
    for message in messages:
        for call in (getattr(message, "tool_calls", None) or []):
            found = True
            print({"tool": call.get("name"), "arguments": call.get("args")})
    if not found:
        print("No tool calls recorded.")

    print("\\nTOOL RESULTS")
    for message in messages:
        if getattr(message, "type", "") == "tool":
            print("Tool:", getattr(message, "name", "unknown"))
            print("Result:", message.content)

    print("\\nFINAL ANSWER")
    for message in reversed(messages):
        if getattr(message, "type", "") == "ai" and not getattr(message, "tool_calls", None):
            content = message.content
            print(content if isinstance(content, str) else json.dumps(content, default=str))
            return content
    print("No final assistant message found.")
    return None

final_answer = inspect_agent_response(response)

TOOL CALLS
{'tool': 'get_claim_details', 'arguments': {'claim_id': 'CLM-10341'}}
\nTOOL RESULTS
Tool: get_claim_details
Result: {"claim_id": "CLM-10341", "status": "submitted", "rejection_code": null, "billed_amount": 132605.57, "approved_amount": null, "admission_date": "2026-03-11", "discharge_date": "2026-03-19", "submitted_at": "2026-03-11 13:43:00", "settled_at": null, "policy_id": "POL-0317", "plan_code": "CORP_GROUP", "sum_insured": 500000, "room_rent_cap_per_day": 5000, "hospital_name": "Mumbai Care Hospital 11", "hospital_city": "Mumbai"}
\nFINAL ANSWER
Your claim **CLM-10341** is currently **submitted** (not yet approved or paid).  
The total amount billed on the claim is **₹132,605.57**.


## 18. Try multiple customer-service scenarios\nObserve which tools the agent selects. To test another demo member, change the identity in Section 7 and rerun the tool definitions and agent creation cells.

In [0]:
test_questions = [
    "What is the status of claim CLM-10341?",
    "Show me my current policy and sum insured.",
    "What is my room rent cap and copay percentage?",
    "Tell me the directory details for hospital HOS-001.",
    "Can you check claim CLM-10341 and tell me the billed and approved amounts?"
]
for q in test_questions:
    print("\\n" + "=" * 80)
    print("CUSTOMER:", q)
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    inspect_agent_response(result)

\n================================================================================
CUSTOMER: What is the status of claim CLM-10341?
TOOL CALLS
{'tool': 'get_claim_details', 'arguments': {'claim_id': 'CLM-10341'}}
\nTOOL RESULTS
Tool: get_claim_details
Result: {"claim_id": "CLM-10341", "status": "submitted", "rejection_code": null, "billed_amount": 132605.57, "approved_amount": null, "admission_date": "2026-03-11", "discharge_date": "2026-03-19", "submitted_at": "2026-03-11 13:43:00", "settled_at": null, "policy_id": "POL-0317", "plan_code": "CORP_GROUP", "sum_insured": 500000, "room_rent_cap_per_day": 5000, "hospital_name": "Mumbai Care Hospital 11", "hospital_city": "Mumbai"}
\nFINAL ANSWER
**Claim CLM-10341** is currently **submitted**. No rejection code or settlement information is available yet.
\n================================================================================
CUSTOMER: Show me my current policy and sum insured.
TOOL CALLS
{'tool': 'get_member_policies', 'arguments

## 19. Test member isolation\nThis claim ID is from a different sample member than `MEM-0317`. The tool should return the same safe no-match response regardless of whether the claim exists for someone else. This is only a basic application check, not production authorization.

In [0]:
print(get_claim_details.invoke({"claim_id": "CLM-10948"}))

No matching claim was found for the currently verified member. Do not reveal whether it exists for another member.


## 20. Enable MLflow tracing\nMLflow Tracing helps inspect application execution. LangChain autologging may capture nested model/tool spans depending on installed integration versions. The explicit trace below records a top-level request. **Privacy:** traces can contain prompts and tool outputs; redact sensitive values and restrict trace access in real deployments.

In [0]:
try:
    mlflow.langchain.autolog()
    print("LangChain autolog enabled.")
except Exception as e:
    print("Autolog setup issue:", str(e)[:1000])
    print("Explicit tracing below can still be used.")

LangChain autolog enabled.


In [0]:
@mlflow.trace(name="customer_service_agent_request")
def run_customer_service_agent(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for message in reversed(result.get("messages", [])):
        if getattr(message, "type", "") == "ai" and not getattr(message, "tool_calls", None):
            content = message.content
            return content if isinstance(content, str) else json.dumps(content, default=str)
    return "The agent did not return a final answer."

with mlflow.start_run(run_name="customer_service_claim_status"):
    mlflow.log_param("model_endpoint", MODEL_ENDPOINT)
    mlflow.log_param("tool_count", len(customer_service_tools))
    print(run_customer_service_agent("Please explain the status of claim CLM-10341."))

**Claim CLM‑10341 – Status Overview**

- **Current status:** **Submitted** – The claim has been received by the system but has not yet been processed for approval or denial.  
- **Rejection code:** None – The claim has not been rejected, so no specific rejection reason applies.  
- **Next steps:** The claim will move to the adjudication phase where the insurer reviews the medical services, verifies coverage under the member’s plan, and determines the payable amount.  

If you need more details (e.g., expected processing time, required documents, or the next action you should take), let me know!


Trace(trace_id='tr-bab164b8271e48a6b40b5554b9616bc8')

## 21. Inspect the MLflow trace\nOpen the MLflow experiment from Section 4. Find the `customer_service_claim_status` run and inspect its trace. Depending on integration compatibility, it may show the top-level request, model calls, tool calls/results, inputs/outputs, and timing. If only the top-level span appears, inspect actual tool calls with Section 17; nested span visibility depends on the installed integrations.

## 22. Basic evaluation\nThis is a smoke test, not a semantic quality evaluation. It records answers and response rate. A full evaluation should also test factual correctness, groundedness, tool selection, refusal behavior, and member isolation.

In [0]:
evaluation_cases = [
    ("E01", "What is the status of claim CLM-10341?"),
    ("E02", "What is my current plan and sum insured?"),
    ("E03", "What is the room rent cap on my plan?"),
    ("E04", "Give me the directory details for hospital HOS-001.")
]
evaluation_rows = []

with mlflow.start_run(run_name="customer_service_smoke_evaluation"):
    mlflow.log_param("model_endpoint", MODEL_ENDPOINT)
    mlflow.log_param("evaluation_case_count", len(evaluation_cases))
    for test_id, question in evaluation_cases:
        answer = run_customer_service_agent(question)
        evaluation_rows.append({
            "test_id": test_id,
            "question": question,
            "answer": answer,
            "response_returned": bool(answer and answer.strip())
        })
    evaluation_df = pd.DataFrame(evaluation_rows)
    display(evaluation_df)
    response_rate = float(evaluation_df["response_returned"].mean())
    mlflow.log_metric("response_rate", response_rate)
    print("Response rate:", response_rate)

test_id,question,answer,response_returned
E01,What is the status of claim CLM-10341?,"**Claim CLM‑10341** is currently **submitted**. No rejection code has been issued, and the claim has not yet been approved or settled.",true
E02,What is my current plan and sum insured?,"Your current policy is **CORP_GROUP** (a group plan). - **Sum insured:** ₹500,000 - **Policy validity:** Starts 29 Dec 2024 (no end date listed).",true
E03,What is the room rent cap on my plan?,"The room rent cap on your plan is **₹5,000 per day**.",true
E04,Give me the directory details for hospital HOS-001.,**Hospital Directory – HOS-001** - **Name:** Mumbai Care Hospital 01 - **City:** Mumbai - **City Tier:** Metro - **State:** Maharashtra,true


Response rate: 1.0


[Trace(trace_id='tr-b8ea7892f1d2552b33b221baf7a9a2bd'), Trace(trace_id='tr-e7093a8f07062e50a78497ddea153a75'), Trace(trace_id='tr-c73ba9e1a5812685abdd95dadb89e0b9'), Trace(trace_id='tr-8bf8aa866009be0e3ca6460310d996e2')]

## 23. Databricks vs. framework responsibilities

| Capability | Technology | What it does |
|---|---|---|
| Governed data | Unity Catalog + Spark | Tables, permissions, queries |
| Model hosting | Databricks Model Serving | Serves the LLM |
| Agent orchestration | LangChain | Tool selection and message loop |
| Business operations | Python + Spark | Controlled lookups |
| Policy retrieval | Databricks AI Search (optional) | Relevant policy chunks |
| Tracing/experiments | MLflow | Runs, metrics, traces |

**Client explanation:** "Databricks is our governed data and AI platform. Claims data remains in Unity Catalog, the LLM is served through Model Serving, Python functions expose controlled operations, LangChain orchestrates the agent, and MLflow provides observability."

## 24. Production considerations

1. **Authentication:** verify the signed-in customer outside the LLM. Derive member ID from a trusted session, not chat text.
2. **Authorization:** enforce row-level access in the backend and Unity Catalog. A system prompt is not a security boundary.
3. **Least privilege:** grant only the minimum required read permissions.
4. **Data minimization:** avoid exposing national IDs, phone numbers, email addresses, or full discharge summaries unless justified and authorized.
5. **SQL safety:** use constrained queries; do not allow unrestricted LLM-generated SQL for customer access.
6. **Policy grounding:** retrieve policy passages for wording questions; structured plan fields may not contain every clause.
7. **Human escalation:** disputed claims, complaints, coverage decisions, and ambiguous cases need an authorized human workflow.
8. **Observability:** redact sensitive values from prompts/traces/logs and restrict experiment access.
9. **Evaluation:** test correctness, hallucinations, tool selection, member isolation, and error handling.
10. **Deployment:** put the app behind an authenticated API/interface. A notebook is for development/demo, not a public customer endpoint.

## 25. Troubleshooting

- **429 / rate limit:** serving endpoint may be throttling requests. Wait, reduce repeated calls, or choose another enabled endpoint.
- **Endpoint not found / permission denied:** confirm the exact endpoint name and your permission to query it.
- **No matching claim:** check whether it belongs to `DEMO_VERIFIED_MEMBER_ID`. The tool intentionally hides whether another member's claim exists.
- **Table not found / permission denied:** verify table names and Unity Catalog `SELECT` privileges.
- **LangChain import or `create_agent` error:** rerun installation/restart and execute cells in order; avoid mixing incompatible LangChain/LangGraph versions.
- **No nested MLflow tool spans:** inspect the returned messages in Section 17. Nested spans depend on integration compatibility.

## 26. Suggested client demonstration

1. Explain that claims data is in Unity Catalog.
2. Show the five analytics tables and join validation.
3. Ask: "What is the status of my claim CLM-10341?"
4. Show the agent selecting `get_claim_details`.
5. Show the tool result and natural-language answer.
6. Ask for plan benefits and show the policy lookup.
7. Open MLflow and inspect the request trace.
8. Explain authentication, row-level permissions, redaction, and human escalation.

**Final takeaway:** the model interprets and communicates; controlled tools retrieve facts; Unity Catalog governs data access; MLflow helps developers inspect and evaluate the application.